# Downstream materials-informatics evaluation

This notebook evaluates segmentation-derived γ′ morphology together with composition and processing metadata for the 99 matched records used in the manuscript.

The analysis includes:
1. provisional composition-and-processing group construction;
2. group-disjoint nested cross-validation;
3. controlled feature ablations;
4. direct segmentation-count comparison; and
5. an acquisition-proxy diagnostic for recorded γ′ size.

Recorded γ′ size was measured using Fiji. It is excluded from the principal prediction targets because it numerically matches image scale in 96 of 99 records, indicating a strong acquisition-associated dependency. The grouping variable is a provisional proxy for shared material conditions and does not establish physical specimen identity.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATASET = ROOT / "data" / "downstream" / "manuscript_analysis_dataset.csv"
BASIC = ROOT / "data" / "morphology" / "detectron2_NEW110_morphology_features_99samples.csv"
EXPANDED = ROOT / "data" / "morphology" / "expanded_morphology_features_NEW110_99samples.csv"
TABLES = ROOT / "results"
TABLES.mkdir(exist_ok=True)

for label, path in [
    ("metadata", DATASET),
    ("basic morphology", BASIC),
    ("expanded morphology", EXPANDED),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {label}: {path}")


In [ ]:
import os, json, hashlib, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from IPython.display import display

BASE = '/content/drive/MyDrive/PHD/Isaac_new/new_file2/SimCLR_and_Detectron2_Prediction'
DATASET = os.path.join(BASE, 'alloy_dataset_area_derived_APPROVED.csv')
OLD_TABLES = os.path.join(BASE, 'experiment3_NEW110_MORPHOLOGY_NESTEDCV_PROXYSAFE', 'tables')
BASIC = os.path.join(OLD_TABLES, 'detectron2_NEW110_morphology_features_99samples.csv')
EXPANDED = os.path.join(OLD_TABLES, 'expanded_morphology_features_NEW110_99samples.csv')
OUT = os.path.join(BASE, 'experiment3_APPROVED_DATA_GROUPCV_POINT2_FINAL')
TABLES = os.path.join(OUT, 'tables')
os.makedirs(TABLES,exist_ok=True)
for label,path in [('metadata',DATASET),('NEW110 basic morphology',BASIC),('NEW110 expanded morphology',EXPANDED)]:
    print(label, os.path.exists(path), path)
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Missing {label}: {path}. Upload the approved dataset or generate NEW110 morphology with the original notebook first.')
print('Outputs:',TABLES)

In [ ]:
def key(v):
    return os.path.splitext(os.path.basename(str(v).replace('\\','/')))[0].strip().lower()
meta=pd.read_csv(DATASET); basic=pd.read_csv(BASIC); expanded=pd.read_csv(EXPANDED)
for name,df in [('metadata',meta),('basic',basic),('expanded',expanded)]:
    if name=='metadata': source='file'
    elif 'file_key' in df: source='file_key'
    elif 'file' in df: source='file'
    else: raise KeyError(f'{name}: missing file/file_key')
    df['file_key']=df[source].map(key)
    if df.file_key.duplicated().any(): raise AssertionError(f'Duplicate {name} keys')
    print(name,df.shape,'sample keys',df.file_key.head(2).tolist())
assert len(meta)==99, f'Expected 99 rows; got {len(meta)}'
assert set(meta.file_key)==set(basic.file_key)==set(expanded.file_key), 'Mismatch among the three datasets; do not continue.'
# Remove redundant file columns before merging, keep only NEW110 feature families.
basic_cols=[x for x in basic if x.startswith('detectron2_') and pd.api.types.is_numeric_dtype(basic[x])]
expanded_cols=[x for x in expanded if x.startswith('advanced_') and pd.api.types.is_numeric_dtype(expanded[x])]
assert basic_cols and expanded_cols, (basic_cols,expanded_cols)
merged=meta.merge(basic[['file_key']+basic_cols],on='file_key',validate='one_to_one').merge(expanded[['file_key']+expanded_cols],on='file_key',validate='one_to_one')
morph=sorted(set(basic_cols+expanded_cols))
composition=sorted(x for x in meta if x.startswith('alloy.composition.') and pd.api.types.is_numeric_dtype(meta[x]))
processing=sorted(x for x in meta if x.startswith('image_properties.processing.') and pd.api.types.is_numeric_dtype(meta[x]))
materials=composition+processing
assert composition and processing and morph
# Drop features that are constant or all missing; imputation occurs ONLY inside each training fold.
def usable(cols):
    return [x for x in cols if merged[x].notna().sum()>0 and merged[x].nunique(dropna=True)>1]
materials=usable(materials); morph=usable(morph)
print('Materials features:',len(materials),'morphology features:',len(morph))
print('Excluded constant/all-missing material features:',sorted(set(composition+processing)-set(materials)))
TARGETS=['features[0].fraction','microstructure[0].count','microstructure[1].total area(nm²)',
         'microstructure[2].Average area(nm²)','microstructure[3].percentage area']
assert all(x in merged for x in TARGETS)
assert 'features[0].size(nm)' not in TARGETS
assert not set(materials+morph)&set(TARGETS+['features[0].size(nm)','image_properties.img_scale(nm)'])
print('Targets:',TARGETS)
# Exact recorded composition + processing grouping. Hash is a label, not a physical specimen identifier.
group_cols=sorted(composition+processing)
def token(v):
    if pd.isna(v): return '<MISSING>'
    try: return format(float(v),'.12g')
    except (ValueError,TypeError): return str(v).strip()
keys=merged[group_cols].apply(lambda r:json.dumps([token(x) for x in r]),axis=1)
merged['group']=keys.map(lambda s:hashlib.sha256(s.encode()).hexdigest()[:16])
gs=merged.groupby('group').size()
print('Provisional groups:',len(gs),'repeated sizes:',sorted(gs[gs>1].tolist()))
# assert len(gs)==96 and sorted(gs[gs>1].tolist())==[2,3], 'Grouping changed; inspect metadata before interpreting comparisons.'
merged[['file_key','group']+group_cols].to_csv(os.path.join(TABLES,'provisional_group_audit.csv'),index=False)
merged[['file_key']+TARGETS].to_csv(os.path.join(TABLES,'target_snapshot.csv'),index=False)


## A. Group-disjoint nested model selection

For each target, the outer 5-fold split groups identical recorded composition + processing together. Within each outer training set, **4-fold GroupKFold** selects model family, feature strategy, and hyperparameters. Only the winning fitted model predicts the held-out outer fold. The outer test fold is never used for selection. Models: Ridge, Random Forest and ExtraTrees (same family subset for all targets). This is a **new run**, not directly identical to the earlier five-model experiment.

In [ ]:
FEATURE_SETS={
 'composition_processing':materials,
 'morphology_only':morph,
 'composition_processing_plus_morphology':sorted(set(materials+morph)),
}
# Keep the candidate set small enough to run in Colab; all comparisons use identical outer splits.
CANDIDATES={
 'Ridge': (Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',Ridge())]),
           {'model__alpha':[0.1,1.,10.,100.]}),
 'RandomForest': (Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestRegressor(random_state=42,n_jobs=-1))]),
           {'model__n_estimators':[200], 'model__max_depth':[None,8], 'model__min_samples_leaf':[1,3]}),
 'ExtraTrees': (Pipeline([('imputer',SimpleImputer(strategy='median')),('model',ExtraTreesRegressor(random_state=42,n_jobs=-1))]),
           {'model__n_estimators':[200], 'model__max_depth':[None,8], 'model__min_samples_leaf':[1,3]})
}
NESTED_FOLDS=[]; NESTED_OOF=[]; NESTED_SELECTION=[]
for target in TARGETS:
    base=merged.loc[merged[target].notna()].reset_index(drop=True)
    y=base[target].astype(float).to_numpy(); groups=base.group.to_numpy()
    assert len(np.unique(groups))>=5
    for fold,(tr,te) in enumerate(GroupKFold(5).split(base,y,groups),1):
        assert not (set(groups[tr]) & set(groups[te]))
        candidates=[]
        for strategy,cols in FEATURE_SETS.items():
            X=base[cols].apply(pd.to_numeric,errors='coerce')
            for model_name,(pipe,grid) in CANDIDATES.items():
                inner=GroupKFold(4)
                search=GridSearchCV(pipe,grid,cv=inner,scoring='neg_mean_squared_error',n_jobs=-1,refit=True)
                # Pass groups to GridSearchCV so inner CV is also group-disjoint.
                search.fit(X.iloc[tr],y[tr],groups=groups[tr])
                record=dict(target=target,outer_fold=fold,strategy=strategy,model=model_name,
                            inner_neg_MSE=float(search.best_score_),params=str(search.best_params_))
                NESTED_SELECTION.append(record)
                candidates.append((search.best_score_,strategy,model_name,search.best_estimator_,cols))
        best=max(candidates,key=lambda t:t[0]); _,strategy,model_name,estimator,cols=best
        pred=estimator.predict(base.iloc[te][cols].apply(pd.to_numeric,errors='coerce'))
        NESTED_FOLDS.append(dict(target=target,outer_fold=fold,strategy=strategy,model=model_name,
                                 n_train=len(tr),n_test=len(te),group_overlap=0,
                                 R2=r2_score(y[te],pred),MAE=mean_absolute_error(y[te],pred),
                                 RMSE=np.sqrt(mean_squared_error(y[te],pred))))
        for j,p in zip(te,pred):
            NESTED_OOF.append(dict(target=target,outer_fold=fold,file_key=base.loc[j,'file_key'],
                                   group=groups[j],actual=float(y[j]),predicted=float(p),
                                   selected_strategy=strategy,selected_model=model_name))
        print(target,'fold',fold,strategy,model_name,'R2',round(r2_score(y[te],pred),3))
folds=pd.DataFrame(NESTED_FOLDS); oof=pd.DataFrame(NESTED_OOF)
summary=folds.groupby('target',as_index=False).agg(R2_outer_mean=('R2','mean'),R2_outer_sd=('R2','std'),
    MAE_outer_mean=('MAE','mean'),RMSE_outer_mean=('RMSE','mean'))
pooled=oof.groupby('target').apply(lambda z:pd.Series(dict(R2_pooled_OOF=r2_score(z.actual,z.predicted),
    MAE_pooled_OOF=mean_absolute_error(z.actual,z.predicted),
    RMSE_pooled_OOF=np.sqrt(mean_squared_error(z.actual,z.predicted)))),include_groups=False).reset_index()
summary=summary.merge(pooled,on='target')
for name,df in [('nested_cv_summary.csv',summary),('nested_cv_outer_folds.csv',folds),
                ('nested_cv_oof_predictions.csv',oof),('nested_cv_inner_selection.csv',pd.DataFrame(NESTED_SELECTION))]:
    df.to_csv(os.path.join(TABLES,name),index=False); print('Saved',name)
display(summary)

## B. Point 2 — fixed-strategy ablation and direct count measurement baseline

All strategies use identical outer GroupKFold partitions and **Ridge with inner group-disjoint tuning**. The direct segmentation count baseline is a measurement-agreement comparison, not independent prediction. The restricted count ablation removes count and density features; remaining morphology features may still encode count indirectly.

In [ ]:
count_target='microstructure[0].count'
count_related=[x for x in morph if 'count' in x.lower() or 'density' in x.lower()]
restricted=[x for x in morph if x not in count_related]
count_sets=dict(FEATURE_SETS)
count_sets['morphology_without_count_density']=restricted
count_sets['combined_without_count_density']=sorted(set(materials+restricted))
ABL_FOLDS=[]; ABL_OOF=[]
for target in TARGETS:
    base=merged.loc[merged[target].notna()].reset_index(drop=True)
    y=base[target].astype(float).to_numpy(); groups=base.group.to_numpy()
    strategies=count_sets if target==count_target else FEATURE_SETS
    for fold,(tr,te) in enumerate(GroupKFold(5).split(base,y,groups),1):
        assert set(groups[tr]).isdisjoint(set(groups[te]))
        for strategy,cols in [('training_mean_baseline',[])]+list(strategies.items()):
            if not cols:
                pred=np.full(len(te),np.mean(y[tr])); alpha=np.nan
            else:
                pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',Ridge())])
                search=GridSearchCV(pipe,{'model__alpha':[.1,1.,10.,100.]},cv=GroupKFold(4),
                                    scoring='neg_mean_squared_error',n_jobs=-1)
                X=base[cols].apply(pd.to_numeric,errors='coerce')
                search.fit(X.iloc[tr],y[tr],groups=groups[tr])
                pred=search.predict(X.iloc[te]);alpha=search.best_params_['model__alpha']
            ABL_FOLDS.append(dict(target=target,outer_fold=fold,strategy=strategy,alpha=alpha,
                                  R2=r2_score(y[te],pred),MAE=mean_absolute_error(y[te],pred),
                                  RMSE=np.sqrt(mean_squared_error(y[te],pred)),group_overlap=0))
            for j,p in zip(te,pred):
                ABL_OOF.append(dict(target=target,outer_fold=fold,strategy=strategy,file_key=base.loc[j,'file_key'],
                                    actual=float(y[j]),predicted=float(p)))
af=pd.DataFrame(ABL_FOLDS); ao=pd.DataFrame(ABL_OOF)
ass=af.groupby(['target','strategy'],as_index=False).agg(R2_outer_mean=('R2','mean'),R2_outer_sd=('R2','std'),
    MAE_outer_mean=('MAE','mean'),RMSE_outer_mean=('RMSE','mean'))
ap=ao.groupby(['target','strategy']).apply(lambda z:pd.Series(dict(R2_pooled_OOF=r2_score(z.actual,z.predicted),
    MAE_pooled_OOF=mean_absolute_error(z.actual,z.predicted),
    RMSE_pooled_OOF=np.sqrt(mean_squared_error(z.actual,z.predicted)))),include_groups=False).reset_index()
ass=ass.merge(ap,on=['target','strategy'])
direct=[]
for col in count_related:
    if 'count' not in col.lower():continue
    z=merged[[count_target,col]].apply(pd.to_numeric,errors='coerce').dropna()
    if len(z):direct.append(dict(segmentation_feature=col,n=len(z),pearson_r=z[count_target].corr(z[col]),
        direct_MAE=mean_absolute_error(z[count_target],z[col]),
        direct_RMSE=np.sqrt(mean_squared_error(z[count_target],z[col]))))
for name,df in [('feature_ablation_summary.csv',ass),('feature_ablation_outer_folds.csv',af),
                ('feature_ablation_oof_predictions.csv',ao),('direct_count_measurement_baselines.csv',pd.DataFrame(direct))]:
    df.to_csv(os.path.join(TABLES,name),index=False);print('Saved',name)
display(ass);display(pd.DataFrame(direct))

## C. Acquisition-proxy diagnostic for recorded γ′ size

Recorded γ′ size was measured using Fiji but numerically matches image scale in 96 of 99 records. This diagnostic compares models with and without image-scale metadata to quantify the strength of that acquisition-associated relationship. These results are diagnostic associations and are not interpreted as independent particle-size predictions.

In [ ]:
scale='image_properties.img_scale(nm)'; size='features[0].size(nm)'
if scale in merged and size in merged:
    z=merged[[scale,size,'file_key']].copy().dropna()
    z['equal']=np.isclose(z[scale],z[size],rtol=0,atol=1e-8)
    print('Identical recorded scale/size:',z['equal'].sum(),'of',len(z))
    display(z.loc[~z['equal']])
    z.to_csv(os.path.join(TABLES,'size_scale_dependency_diagnostic.csv'),index=False)
    # The size target is not an independently validated scientific outcome.
    diag=[]
    raw=[x for x in meta if pd.api.types.is_numeric_dtype(meta[x]) and x not in
         set(TARGETS+[size]) and not x.startswith('creep')]
    for label,cols in [('materials_only',materials),('materials_plus_scale',materials+[scale])]:
        cols=sorted(set(cols))
        X=merged[cols].apply(pd.to_numeric,errors='coerce');y=merged[size].to_numpy(dtype=float)
        pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',Ridge(alpha=10))])
        scores=cross_val_score(pipe,X,y,cv=KFold(5,shuffle=True,random_state=42),scoring='r2')
        diag.append(dict(strategy=label,mean_R2=float(np.mean(scores)),std_R2=float(np.std(scores,ddof=1)),
                         caveat='Diagnostic only: acquisition-associated size-scale dependency'))
    pd.DataFrame(diag).to_csv(os.path.join(TABLES,'gamma_prime_size_proxy_diagnostic.csv'),index=False)
    display(pd.DataFrame(diag))

## Outputs

The notebook writes the group-disjoint nested-CV, feature-ablation, direct-count and acquisition-proxy result tables to the repository `results` directory. The analysis does not establish specimen-level independence, unseen-alloy generalisation, or end-to-end independent validation of the segmentation extractor.
